In [146]:
print("OK")

OK


In [147]:
%pwd

'g:\\'

In [148]:
import os
os.chdir("../")

In [149]:
%pwd

'g:\\'

In [150]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [151]:
#Extract PDF
def load_pdf(data):
    loader = DirectoryLoader(data,
                             glob="*.pdf",
                             loader_cls=PyPDFLoader)
    docs = loader.load()
    return docs

In [152]:
from pathlib import Path

DATA_DIR = Path(r"G:/AAA_Personal/Route_Ranger/LLM_Route_Ranger/data")
extract_data = load_pdf(DATA_DIR)

In [153]:
extract_data

[Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.0 (Windows)', 'creationdate': '2025-03-13T16:29:54-04:00', 'moddate': '2025-03-13T16:31:59-04:00', 'trapped': '/False', 'source': 'G:\\AAA_Personal\\Route_Ranger\\LLM_Route_Ranger\\data\\Travel-Guide-Western-Canada-2025.pdf', 'total_pages': 166, 'page': 0, 'page_label': '1'}, page_content='TRAVEL GUIDE\nWestern\nCanada\n@Shutterstock_1517645066\nTOP 5 ROAD TRIPS IN\nWESTERN CANADA P.09\nThe best tours to discover the Western Canada  —\nGUIDE TO 25 MUST SEE \nDESTINATIONS P.13\nWhat to do\nWhere to eat\nTourist maps\n —\nPRACTICAL INFORMATION P.141\nHow to protect yourself from mosquitoes\nSouvenirs to bring back\nDriving in Canada\nDriving an automatic transmission\nTraveling with a motorhome'),
 Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.0 (Windows)', 'creationdate': '2025-03-13T16:29:54-04:00', 'moddate': '2025-03-13T16:31:59-04:00', 'trapped': '/False',

In [154]:
len(extract_data)

166

In [155]:
from typing import List
from langchain.schema import Document

def filter_text(docs: List[Document]) -> List[Document]:
    """
    Given a list of documents, 
    return a list of documents where the text has been filtered out
    with only 'source' in metadata and original page_content.
    """

    filter_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        filter_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src},
            )
        )
    return filter_docs

In [156]:
filter_docs = filter_text(extract_data)

In [157]:
filter_docs

[Document(metadata={'source': 'G:\\AAA_Personal\\Route_Ranger\\LLM_Route_Ranger\\data\\Travel-Guide-Western-Canada-2025.pdf'}, page_content='TRAVEL GUIDE\nWestern\nCanada\n@Shutterstock_1517645066\nTOP 5 ROAD TRIPS IN\nWESTERN CANADA P.09\nThe best tours to discover the Western Canada  —\nGUIDE TO 25 MUST SEE \nDESTINATIONS P.13\nWhat to do\nWhere to eat\nTourist maps\n —\nPRACTICAL INFORMATION P.141\nHow to protect yourself from mosquitoes\nSouvenirs to bring back\nDriving in Canada\nDriving an automatic transmission\nTraveling with a motorhome'),
 Document(metadata={'source': 'G:\\AAA_Personal\\Route_Ranger\\LLM_Route_Ranger\\data\\Travel-Guide-Western-Canada-2025.pdf'}, page_content='-\nTravel Guide \nwritten by Canadians.\nAuthentik Canada is a Canadian travel agency specializing in the organization of \nroad trips for families and couples.\nwww.AuthentikCanada.com'),
 Document(metadata={'source': 'G:\\AAA_Personal\\Route_Ranger\\LLM_Route_Ranger\\data\\Travel-Guide-Western-Canada-

In [158]:
#Split the document to smaller chunks
def text_split(filter_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,   #tokens
        chunk_overlap=20,
    )
    texts_chunk = text_splitter.split_documents(filter_docs)
    return texts_chunk

In [159]:
texts_chunk = text_split(filter_docs)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 504


In [160]:
texts_chunk

[Document(metadata={'source': 'G:\\AAA_Personal\\Route_Ranger\\LLM_Route_Ranger\\data\\Travel-Guide-Western-Canada-2025.pdf'}, page_content='TRAVEL GUIDE\nWestern\nCanada\n@Shutterstock_1517645066\nTOP 5 ROAD TRIPS IN\nWESTERN CANADA P.09\nThe best tours to discover the Western Canada  —\nGUIDE TO 25 MUST SEE \nDESTINATIONS P.13\nWhat to do\nWhere to eat\nTourist maps\n —\nPRACTICAL INFORMATION P.141\nHow to protect yourself from mosquitoes\nSouvenirs to bring back\nDriving in Canada\nDriving an automatic transmission\nTraveling with a motorhome'),
 Document(metadata={'source': 'G:\\AAA_Personal\\Route_Ranger\\LLM_Route_Ranger\\data\\Travel-Guide-Western-Canada-2025.pdf'}, page_content='-\nTravel Guide \nwritten by Canadians.\nAuthentik Canada is a Canadian travel agency specializing in the organization of \nroad trips for families and couples.\nwww.AuthentikCanada.com'),
 Document(metadata={'source': 'G:\\AAA_Personal\\Route_Ranger\\LLM_Route_Ranger\\data\\Travel-Guide-Western-Canada-

In [161]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace Embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name)
    return embeddings

embeddings = download_embeddings()

In [162]:
embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [163]:
vector = embeddings.embed_query("Hello Data")
vector

[-0.008575072512030602,
 0.056509315967559814,
 0.027168620377779007,
 0.046327948570251465,
 -0.09833744168281555,
 -0.09101816266775131,
 0.10453861206769943,
 -0.01503903977572918,
 -0.04115809127688408,
 -0.022859493270516396,
 0.07448923587799072,
 -0.007406935095787048,
 0.024353045970201492,
 -0.05445876345038414,
 0.007243032567203045,
 0.04555669054389,
 0.055400967597961426,
 -0.0744568407535553,
 -0.11482883244752884,
 0.012796051800251007,
 -0.02966075763106346,
 0.019233660772442818,
 -0.04182029888033867,
 0.0463297963142395,
 0.009186812676489353,
 0.07419151812791824,
 0.015606498345732689,
 0.02183099091053009,
 0.06408178806304932,
 -0.08042002469301224,
 0.03135664761066437,
 -0.002854500664398074,
 0.11842701584100723,
 0.02752244472503662,
 0.00983039103448391,
 -0.0011753452708944678,
 -0.0029793507419526577,
 -0.000506057869642973,
 -0.05223087593913078,
 0.04674738645553589,
 -0.0012295549968257546,
 -0.055244483053684235,
 -0.0248375553637743,
 0.02069182135164

In [164]:
print("Vector length:", len(vector))

Vector length: 384


In [165]:
from dotenv import load_dotenv
import os
load_dotenv()

False

In [166]:
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [167]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [168]:
pc

In [ ]:
from pinecone import ServerlessSpec

index_name = "route-ranger"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension = 384, # dimensions of the embeddings
        metric = "cosine", # consine similarity
        spec = ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [170]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embeddings,
    index_name=index_name
)

In [171]:
#Load Existing index

from langchain_pinecone import PineconeVectorStore
#Embved each chunk and upsert the embeddings into the Pinecone index
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings 
)

# Add more Data to the existing Pinecone index

### Example


In [172]:
dswith = Document(
    page_content="I want to sleep",
    metadata={"source": "personal_care"}
)

In [173]:
docsearch.add_documents(documents=[dswith])

['c424a7d5-6721-4078-9245-b3b5b724ab02']

In [174]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [175]:
retrieved_docs = retriever.invoke("Calgary places recomendendations?")
retrieved_docs

[Document(id='48530e43-6255-4233-a2e4-48026838f3ea', metadata={'source': 'G:\\AAA_Personal\\Route_Ranger\\LLM_Route_Ranger\\data\\Travel-Guide-Western-Canada-2025.pdf'}, page_content='in an authentic setting, with a \npanoramic view of the park! \nRIVER-CAFE.COM\nDESTINATIONS\n© DollaPhotoClub, Jeff\nCALGARY BY NIGHT'),
 Document(id='587deb59-c45a-499e-bb39-6a3dcf8848c9', metadata={'source': 'G:\\AAA_Personal\\Route_Ranger\\LLM_Route_Ranger\\data\\Travel-Guide-Western-Canada-2025.pdf'}, page_content='in an authentic setting, with a \npanoramic view of the park! \nRIVER-CAFE.COM\nDESTINATIONS\n© DollaPhotoClub, Jeff\nCALGARY BY NIGHT'),
 Document(id='5ae8d57b-e595-4fe3-8079-59305376f2a2', metadata={'source': 'G:\\AAA_Personal\\Route_Ranger\\LLM_Route_Ranger\\data\\Travel-Guide-Western-Canada-2025.pdf'}, page_content='in an authentic setting, with a \npanoramic view of the park! \nRIVER-CAFE.COM\nDESTINATIONS\n© DollaPhotoClub, Jeff\nCALGARY BY NIGHT')]

In [176]:
pip install langchain-google-genai  # the openai is not free and I am poor.

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [177]:
from langchain_google_genai import ChatGoogleGenerativeAI

chatModel = ChatGoogleGenerativeAI(model="gemini-2.5-pro")

In [178]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt = (
    "You are a friendly and knowledgeable travel assistant specializing in Western Canada."
    "Answer questions using only the information provided below."
    "If something is not included in the information, simply reply with 'I do not know.'"
    "Stay focused on Western Canada travel and avoid unrelated topics."
    " Do not invent information—only answer using the given context."
    "Keep responses concise, with a maximum of five sentences."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [180]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [181]:
response = rag_chain.invoke({"input": "Calgary places recomendendations?"})
print(response["answer"])

The River-Cafe is a recommended destination in Calgary. It is described as being in an authentic setting with a panoramic view of the park. You can find more information at RIVER-CAFE.COM.


In [182]:
response = rag_chain.invoke({"input": "am I wanna sleep?"})
print(response["answer"])

I don't know.
